# ADA Data Manipulation - Category System

## How to Add New Categories

### Step 1: Update your tariff-hscodes CSV
Add entries with your new category names:
```csv
HS Code,Category
440710,Lumber (old)
440910,Lumber (new)
...
```

### Step 2: Update CATEGORY_CONFIG (Cell 3)
Map each CSV category name to a short column prefix:
```python
CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Lumber (old)': 'LumOld',   # <-- Add new categories
    'Lumber (new)': 'LumNew',   # <-- Add new categories
    ...
}
```

### Step 3: Run all cells
The notebook will automatically:
- Create NAICS sets for each category
- Generate `{Prefix}_B`, `{Prefix}_E` columns (weighted business/employee counts)
- Generate `{Prefix}_1`, `{Prefix}_2`, `{Prefix}_3` columns (percentages for choropleth)
- Export everything to GeoJSON, Shapefile, and CSV

### Output Columns per Category
- `{Prefix}_B` = Weighted number of businesses
- `{Prefix}_E` = Weighted number of employees (by work location)
- `{Prefix}_C` = Weighted number of employees (by residence)
- `{Prefix}_1` = % of all businesses affected
- `{Prefix}_2` = % of all employees affected (by work location)
- `{Prefix}_3` = % of census population in affected jobs (by residence)

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
from tqdm import tqdm
import json
import gc
import time
from datetime import timedelta

## Configuration: Define your tariff categories here

In [3]:
# ============================================================
# CATEGORY CONFIGURATION - Edit this to add/modify categories
# ============================================================

# Path to the tariff-hscodes CSV file
TARIFF_HSCODES_FILE = '../raw/tariff_hs_codes_8_27_2026.csv'

# Category configuration: maps CSV category names to short column prefixes
# Format: 'Category Name in CSV': 'Short_Prefix_for_Columns'

CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Steel': 'Steel',
    'Copper': 'Cop',
    'Energy Mineral': 'Ene',
    'MHDV': 'MHDV',
    # Add new categories here:
    'Lumber (old)': 'LumOld',
    'Lumber (new)': 'LumNew',
    'Dairy': 'Dairy',
    'Alcohol': 'Alcohol',
    'Motor': 'Motor',
    'before August 22': 'before August 22',
    'after August 22': 'after August 22',
    'Section 338': 'Section 338'
}

# Always-included special categories (don't change these unless you know what you're doing)
SPECIAL_CATEGORIES = ['nonCUSMA', 'Total']  # nonCUSMA = goods not covered by CUSMA

print(f"✅ Configured {len(CATEGORY_CONFIG)} tariff categories: {list(CATEGORY_CONFIG.keys())}")
print(f"   Using tariff file: {TARIFF_HSCODES_FILE}")

✅ Configured 14 tariff categories: ['Auto', 'Aluminum', 'Steel', 'Copper', 'Energy Mineral', 'MHDV', 'Lumber (old)', 'Lumber (new)', 'Dairy', 'Alcohol', 'Motor', 'before August 22', 'after August 22', 'Section 338']
   Using tariff file: ../raw/tariff_hs_codes_8_27_2026.csv


# STEP 3: Connecting Tariffed HS Codes, NAICS Codes and respective CUSMA Non-Utilisation Rates via Concordance Table

In [4]:
tariffed = pd.read_csv(TARIFF_HSCODES_FILE, encoding_errors='ignore', dtype={'HS Code': str})

tariffed['HS_Code_6digit'] = (
    tariffed['HS Code']
    .str.replace('.', '', regex=False)
    .str[:6]
)

tariffed = tariffed[['HS_Code_6digit', 'Category']].drop_duplicates()

# Validate that all categories in CSV are in our config
csv_categories = set(tariffed['Category'].dropna().unique())
configured_categories = set(CATEGORY_CONFIG.keys())
unknown_categories = csv_categories - configured_categories

if unknown_categories:
    print(f"⚠️ WARNING: Found categories in CSV not in CATEGORY_CONFIG: {unknown_categories}")
    print("   Add them to CATEGORY_CONFIG or they will be treated as 'nonCUSMA'")
else:
    print(f"✅ All CSV categories are configured: {csv_categories}")

✅ All CSV categories are configured: {'MHDV', 'Section 338', 'Aluminum', 'before August 22', 'Motor', 'Alcohol', 'Steel', 'after August 22', 'Auto', 'Energy Mineral', 'Copper', 'Dairy', 'Lumber (old)', 'Lumber (new)'}


In [5]:
concordance = pd.read_csv('../raw/C616_HS8toNaics6_concord_202505.csv', dtype={'hts10': str})

concordance['HS_Code_6digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:6]
)

concordance['HS_Code_2digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:2]
)

concordance['NAICS'] = concordance['NAICS 6 Code'].astype(str)

concordance = concordance[['HS_Code_6digit', 'NAICS', 'HS_Code_2digit']].drop_duplicates()

print(concordance[concordance['HS_Code_6digit'] == '440311'].head())

     HS_Code_6digit   NAICS HS_Code_2digit
4468         440311  321114             44


In [6]:
util = pd.read_csv('../raw/USMCA Utilization Data.csv')

util['HS_Code_2digit'] = (
    util['HS Classification']
    .astype(str)
    .str[:2]
)

util['nonutil_rate'] = util['USMCA_Nonutilisation_May2025']

util = util[['HS_Code_2digit', 'nonutil_rate']]

Setting the non-utilisation rate for those with sectoral tariffs at 1 reflects the fact that the 35% tariffs on non-CUSMA goods does not apply to the sectoral tariffs and that producers impacted by sectoral tariffs cannot use CUSMA to get their goods tariff-free.

In [7]:
naics_tar = concordance.merge(tariffed, on='HS_Code_6digit', how='left')

counts = naics_tar['HS_Code_6digit'].value_counts().reset_index()
naics_tarc = naics_tar.merge(counts, on='HS_Code_6digit', how='left')

naics_imp = naics_tarc.merge(util, on='HS_Code_2digit', how='left')

# new column non-util chapter getting the nonutil rate! (for before section 338)
naics_imp['nonutil_chapter'] = naics_imp['nonutil_rate'] 

# Making the codes with a sectoral tariff category assigned to have a non-util rate value of 1
# This ensures that when multiplied later, sectoral tariffs do not affect the CUSMA non-utilisation rates to compute the impact of nonCUSMA 35% tariffs
naics_imp.loc[naics_imp['Category'].notna(), 'nonutil_rate'] = 1
naics_imp['Category'] = naics_imp['Category'].fillna('nonCUSMA')
naics_imp

,HS_Code_6digit,NAICS,HS_Code_2digit,Category,count,nonutil_rate,nonutil_chapter
0,010110,112920,01,nonCUSMA,1,0.42,0.42
1,010121,112920,01,nonCUSMA,1,0.42,0.42
2,010129,112920,01,nonCUSMA,1,0.42,0.42
3,010130,112920,01,nonCUSMA,1,0.42,0.42
4,010190,112920,01,nonCUSMA,1,0.42,0.42
...,...,...,...,...,...,...,...
9509,961620,314990,96,nonCUSMA,1,0.84,0.84
9510,961700,332439,96,nonCUSMA,1,0.84,0.84
9511,961800,339990,96,nonCUSMA,1,0.84,0.84
9512,961900,322291,96,nonCUSMA,1,0.84,0.84


# STEP 4: Getting Weights by Province/Territory

Since multiple NAICS codes may contribute to the production of one HS code product, and we do not how much of a part does each NAICS contribute to the whole HS code good production, **thus an assumption is made to divide them equally**. Hence, when each export value is added, it is divided them by the count (how many times does that HS Code get repeated).  

Meanwhile, the non-utilisation rate of CUSMA exemption by each HS Code is first multiplied to the total value of each HS Code export to the US, before divided by the count.

In [8]:
EXPORT_DIR = '../raw/exports'

# Initialize the DataFrame with NAICS data
tariff_exp_val = naics_imp.copy()

# Define all regions to process
provinces = ['NL', 'PEI', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AL', 'BC', 'YK', 'NWT', 'NU']
cols = ['Commodity', 'Value ($)']

for province in provinces:
    # Process Global data
    global_df = pd.read_csv(f'{EXPORT_DIR}/{province}-Global.csv', usecols=cols)
    global_df['HS_Code_6digit'] = global_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    global_df[f'{province}_Global'] = global_df['Value ($)']
    global_df = global_df[['HS_Code_6digit', f'{province}_Global']]
    
    tariff_exp_val = tariff_exp_val.merge(global_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_Global'] = tariff_exp_val[f'{province}_Global'] / tariff_exp_val['count']
    
    # Process US data
    us_df = pd.read_csv(f'{EXPORT_DIR}/{province}-US.csv', usecols=cols)
    us_df['HS_Code_6digit'] = us_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    us_df[f'{province}_US'] = us_df['Value ($)']
    us_df = us_df[['HS_Code_6digit', f'{province}_US']]
    
    tariff_exp_val = tariff_exp_val.merge(us_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_US_raw'] = tariff_exp_val[f'{province}_US'] / tariff_exp_val['count']
    ## NOW HAS THE NON UTIL RATE AS WELL!
    tariff_exp_val[f'{province}_US'] = tariff_exp_val[f'{province}_US_raw'] * tariff_exp_val['nonutil_rate']

# Drop the non-relevant columns
tariff_exp_val = tariff_exp_val.drop(columns=['HS_Code_2digit', 'count', 'nonutil_rate'])

tariff_exp_val = tariff_exp_val.fillna(0)

tariff_exp_val

,HS_Code_6digit,NAICS,Category,nonutil_chapter,NL_Global,NL_US,NL_US_raw,PEI_Global,PEI_US,PEI_US_raw,...,BC_US_raw,YK_Global,YK_US,YK_US_raw,NWT_Global,NWT_US,NWT_US_raw,NU_Global,NU_US,NU_US_raw
0,010110,112920,nonCUSMA,0.42,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,010121,112920,nonCUSMA,0.42,0.0,0.0,0.0,0.0,0.00,0.0,...,43608.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,010129,112920,nonCUSMA,0.42,0.0,0.0,0.0,103341.0,43403.22,103341.0,...,8863555.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,010130,112920,nonCUSMA,0.42,0.0,0.0,0.0,0.0,0.00,0.0,...,16453.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,010190,112920,nonCUSMA,0.42,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9509,961620,314990,nonCUSMA,0.84,0.0,0.0,0.0,0.0,0.00,0.0,...,23452.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9510,961700,332439,nonCUSMA,0.84,0.0,0.0,0.0,0.0,0.00,0.0,...,69403.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9511,961800,339990,nonCUSMA,0.84,0.0,0.0,0.0,0.0,0.00,0.0,...,36336.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9512,961900,322291,nonCUSMA,0.84,0.0,0.0,0.0,0.0,0.00,0.0,...,5790.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Step 1: Aggregate ONLY the necessary sums (Global/US columns)
aggregates = {
    **{f'{province}_Global': (f'{province}_Global', 'sum') for province in provinces},
    **{f'{province}_US': (f'{province}_US', 'sum') for province in provinces},
}

weight_naics = tariff_exp_val.groupby('NAICS', as_index=False).agg(**aggregates)

# Step 2: Compute rates and keep ONLY those columns
for province in provinces:
    global_col = f'{province}_Global'
    us_col = f'{province}_US'
    rate_col = f'{province}'
    
    weight_naics[rate_col] = (
        weight_naics[us_col] / weight_naics[global_col]
    )

# Step 3: Select only naics, naics_2digit, and rate columns
final_columns = ['NAICS'] + [f'{province}' for province in provinces]
weight_naics = weight_naics[final_columns]
weight_naics = weight_naics.fillna(0)

# Result
weight_naics

,NAICS,NL,PEI,NS,NB,QC,ON,MB,SK,AL,BC,YK,NWT,NU
0,111110,0.000000,0.000000,0.000000,0.620000,0.016695,0.057076,0.064938,0.000728,0.024138,0.000000,0.00,0.0,0.00
1,111120,0.620000,0.620000,0.000000,0.124930,0.039934,0.371635,0.094353,0.047928,0.034204,0.028733,0.00,0.0,0.00
2,111130,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.00
3,111140,0.000000,0.000000,0.620000,0.620000,0.045639,0.032567,0.030494,0.038389,0.032376,0.028748,0.00,0.0,0.00
4,111150,0.000000,0.000000,0.000000,0.000000,0.194542,0.074227,0.370000,0.000000,0.370000,0.243865,0.00,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,339920,0.079019,0.227019,0.949896,0.272781,0.862589,0.702178,0.832748,0.873810,0.944934,0.742929,1.00,0.0,0.00
306,339930,0.854647,0.000000,0.945270,1.000000,0.826392,0.591745,0.802942,0.974964,0.877695,0.606863,0.00,0.0,1.00
307,339940,0.000000,0.000000,0.049719,0.831198,0.467514,0.732703,0.859608,0.150000,0.720095,0.691042,0.00,0.0,0.00
308,339950,0.000000,0.150000,0.149934,0.699527,0.748507,0.562703,0.151730,0.604759,0.206830,0.265313,0.15,0.0,0.15


In [10]:
# Categories in force BEFORE Aug 22, 2026 (exact strings from CATEGORY_CONFIG)
IN_FORCE_BEFORE = [
    'Auto', 'Aluminum', 'Steel', 'Copper', 'Energy Mineral', 'MHDV',
    'Lumber (old)', 'Lumber (new)', 'before August 22',
]

# An HS code keeps rate 1.0 if ANY of its tags was in force before Aug 22.
# Otherwise the whole code falls back to its chapter non-utilisation rate.
hs_in_force = set(
    tariff_exp_val.loc[tariff_exp_val['Category'].isin(IN_FORCE_BEFORE), 'HS_Code_6digit']
)

t = tariff_exp_val.copy()
rate_sb = np.where(t['HS_Code_6digit'].isin(hs_in_force), 1.0, t['nonutil_chapter'])

for province in provinces:
    t[f'{province}_US'] = t[f'{province}_US_raw'] * rate_sb

weight_naics_sb = t.groupby('NAICS', as_index=False).agg(**aggregates)
for province in provinces:
    weight_naics_sb[province] = weight_naics_sb[f'{province}_US'] / weight_naics_sb[f'{province}_Global']
weight_naics_sb = weight_naics_sb[['NAICS'] + provinces].fillna(0)

weight_naics_sb

,NAICS,NL,PEI,NS,NB,QC,ON,MB,SK,AL,BC,YK,NWT,NU
0,111110,0.000000,0.000000,0.000000,0.620000,0.016695,0.057076,0.064938,0.000728,0.024138,0.000000,0.00,0.0,0.00
1,111120,0.620000,0.620000,0.000000,0.124930,0.039934,0.371635,0.094353,0.047928,0.034204,0.028733,0.00,0.0,0.00
2,111130,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.00
3,111140,0.000000,0.000000,0.620000,0.620000,0.045639,0.032567,0.030494,0.038389,0.032376,0.028748,0.00,0.0,0.00
4,111150,0.000000,0.000000,0.000000,0.000000,0.194542,0.074227,0.370000,0.000000,0.370000,0.243865,0.00,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,339920,0.079019,0.227019,0.706480,0.243381,0.808154,0.653166,0.582863,0.866177,0.663320,0.587708,1.00,0.0,0.00
306,339930,0.358952,0.000000,0.397013,0.420000,0.347140,0.248730,0.337235,0.409485,0.368632,0.255254,0.00,0.0,0.42
307,339940,0.000000,0.000000,0.049719,0.831198,0.467514,0.732703,0.859608,0.150000,0.720095,0.691042,0.00,0.0,0.00
308,339950,0.000000,0.150000,0.149934,0.699527,0.748507,0.562703,0.151730,0.604759,0.206830,0.265313,0.15,0.0,0.15


In [11]:
hs_ba = tariff_exp_val.loc[tariff_exp_val['Category'] == 'before August 22', 'HS_Code_6digit']
print(tariff_exp_val[tariff_exp_val['HS_Code_6digit'].isin(hs_ba)]['Category'].value_counts())

Category
before August 22    827
after August 22     827
Energy Mineral      315
Steel               308
Auto                137
Aluminum            120
Copper               40
Lumber (new)         30
Motor                27
Section 338          27
MHDV                 24
Lumber (old)          9
Alcohol               1
Name: count, dtype: int64


In [12]:
sb = weight_naics_sb.set_index('NAICS')['ON']
base = weight_naics.set_index('NAICS')['ON']
d = (sb - base).dropna()
print(f"above base: {(d > 1e-12).sum()}   (must be 0)")
print(f"below base: {(d < -1e-12).sum()}")

above base: 0   (must be 0)
below base: 111


# STEP 5: Using filtered NAICS Codes to filter directly-impacted businesses and estimate directly-impacted employees

In [13]:
# DYNAMICALLY CREATE NAICS SETS FOR EACH CATEGORY

# Dictionary to store NAICS codes for each category
category_naics = {}

# Create NAICS sets for each configured category
for csv_name, short_name in CATEGORY_CONFIG.items():
    category_naics[short_name] = set(
        naics_imp[naics_imp['Category'] == csv_name]['NAICS'].unique()
    )
    print(f"  {short_name}: {len(category_naics[short_name])} NAICS codes")

# Special categories (always included)
category_naics['CUSMA'] = set(naics_imp[naics_imp['Category'] == 'nonCUSMA']['NAICS'].unique())
category_naics['Total'] = set(naics_imp['NAICS'].unique())

print(f"\n✅ Created {len(category_naics)} NAICS sets")
print(f"   Total unique NAICS codes: {len(category_naics['Total'])}")

# For backward compatibility, also create individual variables (optional)
total_naics = category_naics['Total']

  Auto: 38 NAICS codes
  Alum: 47 NAICS codes
  Steel: 43 NAICS codes
  Cop: 6 NAICS codes
  Ene: 39 NAICS codes
  MHDV: 5 NAICS codes
  LumOld: 5 NAICS codes
  LumNew: 12 NAICS codes
  Dairy: 13 NAICS codes
  Alcohol: 13 NAICS codes
  Motor: 129 NAICS codes
  before August 22: 120 NAICS codes
  after August 22: 208 NAICS codes
  Section 338: 134 NAICS codes

✅ Created 16 NAICS sets
   Total unique NAICS codes: 310


In [14]:
SCENARIOS = {
    'ScenBefore': ['before August 22', 'nonCUSMA'],
    'ScenAfter':  ['after August 22',  'nonCUSMA'],
}
# if step 1 returned 0, extend each list with the sectoral categories in force

for name, cats in SCENARIOS.items():
    category_naics[name] = set().union(*(
        set(naics_imp.loc[naics_imp['Category'] == c, 'NAICS'].unique()) for c in cats
    ))

In [15]:
print(category_naics['ScenAfter'] == category_naics['Total'])

True


In [16]:
SB, SA = category_naics['ScenBefore'], category_naics['ScenAfter']
S338, nonC = category_naics['Section 338'], category_naics['CUSMA']

print(f"ScenAfter adds {len(SA - SB)} NAICS over ScenBefore")
print(f"S338 NAICS not already non-CUSMA: {len(S338 - nonC)} of {len(S338)}")
print(f"ScenAfter == Total: {SA == category_naics['Total']}")

ScenAfter adds 5 NAICS over ScenBefore
S338 NAICS not already non-CUSMA: 5 of 134
ScenAfter == Total: True


In [17]:
added = category_naics['ScenAfter'] - category_naics['ScenBefore']
print(sorted(added))
print(naics_imp[naics_imp['NAICS'].isin(added)][['NAICS','Category']].drop_duplicates())

['111412', '212393', '312310', '326130', '326160']
       NAICS         Category
942   111412            Motor
943   111412  after August 22
944   111412      Section 338
977   312310            Dairy
978   312310            Motor
979   312310  after August 22
980   312310      Section 338
1419  212393            Motor
1420  212393  after August 22
1421  212393      Section 338
3081  326130            Motor
3082  326130  after August 22
3083  326130      Section 338
3129  326160            Motor
3130  326160  after August 22
3131  326160      Section 338


In [19]:
added = category_naics['ScenAfter'] - category_naics['ScenBefore']
rows = naics_imp[naics_imp['NAICS'].isin(added)]
print(rows[['NAICS','HS_Code_6digit','Category']].drop_duplicates().sort_values('NAICS'))

print(naics_tarc['Category'].isna().sum(), "untagged rows")
print(naics_tarc['Category'].notna().sum(), "tagged rows")

       NAICS HS_Code_6digit         Category
942   111412         121190            Motor
943   111412         121190  after August 22
944   111412         121190      Section 338
1421  212393         250100      Section 338
1420  212393         250100  after August 22
1419  212393         250100            Motor
980   312310         130190      Section 338
990   312310         130219            Motor
991   312310         130219  after August 22
992   312310         130219      Section 338
978   312310         130190            Motor
977   312310         130190            Dairy
979   312310         130190  after August 22
3081  326130         391990            Motor
3082  326130         391990  after August 22
3083  326130         391990      Section 338
3113  326130         392190            Motor
3114  326130         392190  after August 22
3115  326130         392190      Section 338
3130  326160         392330  after August 22
3129  326160         392330            Motor
3131  3261

In [17]:
lum = category_naics['LumOld'] | category_naics['LumNew']
print(f"lumber NAICS not already non-CUSMA: {len(lum - category_naics['CUSMA'])} of {len(lum)}")

lumber NAICS not already non-CUSMA: 0 of 12


In [18]:
# ============================================================
# DYNAMICALLY CREATE COLUMN STRUCTURE
# ============================================================

# Base columns (always present)
col_i = ['DA', 'All_Businesses', 'All_Employees']

# Add columns for each category (Business and Employee counts)
all_category_prefixes = list(CATEGORY_CONFIG.values()) + ['CUSMA', 'Total'] + list(SCENARIOS.keys())
for prefix in all_category_prefixes:
    col_i.extend([f'{prefix}_B', f'{prefix}_E'])

# Add individual NAICS codes as column headers for Est_Employees by NAICS
col_i.extend(sorted(total_naics))

business = pd.DataFrame(columns = col_i)

print(f"✅ Created DataFrame with {len(col_i)} columns")
print(f"   Category columns: {[p for p in all_category_prefixes]}")

✅ Created DataFrame with 349 columns
   Category columns: ['Auto', 'Alum', 'Steel', 'Cop', 'Ene', 'MHDV', 'LumOld', 'LumNew', 'Dairy', 'Alcohol', 'Motor', 'before August 22', 'after August 22', 'Section 338', 'CUSMA', 'Total', 'ScenBefore', 'ScenAfter']


In [19]:
# ============================================================
# MAIN PROCESSING LOOP - DYNAMIC CATEGORIES
# ============================================================

chunk_size = 1_000_000
province_code = {10:'NL',11:'PEI',12:'NS',13:'NB',24:'QC',35:'ON',46:'MB',47:'SK',48:'AL',59:'BC',60:'YK',61:'NWT',62:'NU'}

# melting weights in long form to prepare for a vectorized merge
wlong = (
    weight_naics
    .melt(id_vars='NAICS', var_name='Province', value_name='Rate')
)

wlong = wlong.merge(
    weight_naics_sb.melt(id_vars='NAICS', var_name='Province', value_name='Rate_SB'),
    on=['NAICS', 'Province'], how='outer'
)

# loading necessary data
all_cols = pd.read_csv('../input-data/large_size_data/Dec2022_Estabcounts_byDA.csv', encoding='ISO-8859-1', nrows=1).columns
cols_to_keep = [c for c in all_cols if c != 'Without employees']

# suggesting dtypes for performance purposes
dtype_hint = {
    '1-4':'Int64','5-9':'Int64','10-19':'Int64','20-49':'Int64',
    '50-99':'Int64','100-199':'Int64','200-499':'Int64','500 +':'Int64',
    'Total, with employees':'Int64',
}

total_start = time.time()
chunk_num = 0

# Preparing two empty lists
agg_frames = []       # List for weighted amount of businesses and est employees for each tariff
per_naics_frames = [] # List for total amount of employees for each NAICS code in each ADA --> needed for Step 8 later

# Build the list of aggregation columns dynamically
agg_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    agg_cols.extend([f'{prefix}_B', f'{prefix}_E'])

for chunk in pd.read_csv(
        '../input-data/large_size_data/Dec2022_Estabcounts_byDA.csv',
        encoding='ISO-8859-1',
        chunksize=chunk_size,
        usecols=cols_to_keep,
        dtype=dtype_hint,
    ):
    t0 = time.time()
    chunk_num += 1

    # 1) Filter non-relevant rows
    chunk = chunk[~chunk['NAICS'].isin(['Sub-total, classified', 'Unclassified', 'Total'])].copy()

    # 2) Basic transforms (vectorized)
    # keep NAICS 6-digit as string
    chunk['NAICS'] = chunk['NAICS'].astype(str).str[:6]
    chunk['Business_per_NAICS'] = chunk['Total, with employees'].fillna(0)

    # estimate employees (vectorized)
    chunk['Est_Employees'] = (
        chunk['1-4'].fillna(0) * 3  +
        chunk['5-9'].fillna(0) * 7  +
        chunk['10-19'].fillna(0) * 15 +
        chunk['20-49'].fillna(0) * 35 +
        chunk['50-99'].fillna(0) * 75 +
        chunk['100-199'].fillna(0) * 150 +
        chunk['200-499'].fillna(0) * 350 +
        chunk['500 +'].fillna(0) * 550
    )

    # Province lookup
    # if 'DisseminationAre' isn't numeric, ensure this still works (it uses first two chars)
    chunk['ProvinceCode'] = chunk['DisseminationAre'].astype(str).str[:2].astype(int, errors='ignore')
    chunk['Province'] = pd.Series(chunk['ProvinceCode']).map(province_code)

    # 3) Merge the per-(NAICS, Province) Rate (vectorized, no apply)
    merged = chunk.merge(wlong, how='left', on=['NAICS','Province'])

    # 4) Weighted columns (vectorized)
    merged['Weighted_Business']  = np.ceil(merged['Business_per_NAICS'] * merged['Rate'])
    merged['Weighted_Employees'] = np.ceil(merged['Est_Employees'] * merged['Rate'])

    # 5) DYNAMIC CATEGORY MASKS - Create masks for each category
    s = merged['NAICS']
    wb = merged['Weighted_Business']
    we = merged['Weighted_Employees']

    # 6) the rate with non section 338 weights (original)
    wb_sb = np.ceil(merged['Business_per_NAICS'] * merged['Rate_SB'])
    we_sb = np.ceil(merged['Est_Employees'] * merged['Rate_SB'])
    
    # Apply masks dynamically for each category
    for prefix, naics_set in category_naics.items():
        is_in_category = s.isin(naics_set) if len(naics_set) else pd.Series(False, index=s.index)
        # new
        b, e = (wb_sb, we_sb) if prefix == 'ScenBefore' else (wb, we)
        merged[f'{prefix}_B'] = np.where(is_in_category, b, 0)
        merged[f'{prefix}_E'] = np.where(is_in_category, e, 0)

    # Always aggregate the unweighted totals too
    merged['All_Businesses'] = merged['Business_per_NAICS']
    merged['All_Employees']  = merged['Est_Employees']

    # 6) Chunk-level aggregation in ONE groupby
    by_da = merged.groupby('DisseminationAre', as_index=False)[agg_cols].sum()

    # 7) Generating the data for second list --> number of jobs per NAICS in each DA
    is_total = s.isin(total_naics)
    per_naics = (
        merged.loc[is_total, ['DisseminationAre','NAICS','Est_Employees']]
        .pivot_table(index='DisseminationAre', columns='NAICS', values='Est_Employees',
                     aggfunc='sum', fill_value=0)
        .reset_index()
    )

    # Saving the data into the two different lists in each chunk
    agg_frames.append(by_da)
    per_naics_frames.append(per_naics)

    print(f"Chunk {chunk_num} processed in {time.time()-t0:.2f} sec")

# Combine all chunks together to form one big dataframe
agg_all = pd.concat(agg_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

per_naics_all = pd.concat(per_naics_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

business = agg_all.merge(per_naics_all, on='DisseminationAre', how='left')

business = business.rename(columns={'DisseminationAre':'DA'})

print(f"\n✅ All chunks processed in {time.time()-total_start:.2f} sec")
print(f"   Processed categories: {list(category_naics.keys())}")

business

Chunk 1 processed in 6.08 sec
Chunk 2 processed in 7.24 sec
Chunk 3 processed in 5.79 sec
Chunk 4 processed in 6.05 sec
Chunk 5 processed in 5.60 sec
Chunk 6 processed in 5.37 sec
Chunk 7 processed in 5.41 sec
Chunk 8 processed in 5.37 sec
Chunk 9 processed in 5.55 sec
Chunk 10 processed in 5.46 sec
Chunk 11 processed in 5.51 sec
Chunk 12 processed in 5.53 sec
Chunk 13 processed in 5.46 sec
Chunk 14 processed in 5.47 sec
Chunk 15 processed in 5.49 sec
Chunk 16 processed in 5.50 sec
Chunk 17 processed in 5.56 sec
Chunk 18 processed in 5.48 sec
Chunk 19 processed in 5.50 sec
Chunk 20 processed in 5.53 sec
Chunk 21 processed in 5.52 sec
Chunk 22 processed in 5.53 sec
Chunk 23 processed in 5.53 sec
Chunk 24 processed in 5.63 sec
Chunk 25 processed in 5.63 sec
Chunk 26 processed in 5.56 sec
Chunk 27 processed in 5.60 sec
Chunk 28 processed in 5.56 sec
Chunk 29 processed in 5.62 sec
Chunk 30 processed in 6.12 sec
Chunk 31 processed in 5.68 sec
Chunk 32 processed in 5.61 sec
Chunk 33 processe

,DA,All_Businesses,All_Employees,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,...,337215,337910,337920,339110,339910,339920,339930,339940,339950,339990
0,10000000,170,1006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,10010165,22,266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,10010166,2,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,10010167,6,22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,10010168,5,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55246,62080023,2,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55247,62080024,11,113,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55248,62080025,11,356,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55249,62080026,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


# STEP 6: Regrouping filtered data into ADAs

In [20]:
da = gpd.read_file('../input-data/large_size_data/lda_000b21a_e.shp')
da ['DA'] = da ['DAUID']
da ['DADGUID'] = da ['DGUID']
da = da[['DA', 'DADGUID']]

ada = gpd.read_file('../input-data/large_size_data/lada000b21a_e.shp')
adac = ada.copy()
adac ['ADADGUID'] = adac ['DGUID']
adac = adac[['ADADGUID', 'geometry']]

relation = pd.read_csv('../intermediate/ada_da_relation.csv')

In [21]:
da_relation = da.merge(relation, on='DADGUID', how='left')
full_relation = da_relation.merge(adac, on='ADADGUID', how='left')
full_relation['DA'] = pd.to_numeric(full_relation['DA'], errors='coerce').astype('Int64')

In [22]:
# ============================================================
# AGGREGATE TO ADA LEVEL - DYNAMIC CATEGORIES
# ============================================================

business_merged =(
    business.merge(full_relation, on='DA', how='right')
) 

# Separate numeric columns from geometry
numeric_cols = [col for col in business.columns if col != 'DA']

# Fill NA only for numeric columns, then cast to int64
for col in numeric_cols:
    business_merged[col] = business_merged[col].fillna(0).astype('int64')

# Build aggregation dictionary dynamically
agg_dict = {
    'All_Businesses': ('All_Businesses', 'sum'),
    'All_Employees': ('All_Employees', 'sum'),
    'geometry': ('geometry', 'first')
}

# Add category columns dynamically
for prefix in all_category_prefixes:
    agg_dict[f'{prefix}_B'] = (f'{prefix}_B', 'sum')
    agg_dict[f'{prefix}_E'] = (f'{prefix}_E', 'sum')

# Add dynamic aggregation rules for each NAICS code
for naics in total_naics:
    agg_dict[naics] = (naics, 'sum')

# Perform grouped aggregation
business_grouped = business_merged.groupby('ADADGUID', as_index=False).agg(**agg_dict)

print(f"✅ Aggregated to {len(business_grouped)} ADAs with {len(all_category_prefixes)} category pairs")

business_grouped

✅ Aggregated to 5433 ADAs with 18 category pairs


C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\2482351713.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  business_grouped = business_merged.groupby('ADADGUID', as_index=False).agg(**agg_dict)


,ADADGUID,All_Businesses,All_Employees,geometry,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,...,111130,316210,212114,333130,339950,333413,334210,311351,326191,323116
0,2021S051610010001,219,2910,"MULTIPOLYGON (((8921559.157 2130255.903, 89215...",0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2021S051610010002,66,394,"POLYGON ((8959571.943 2171799.486, 8959576.689...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2021S051610010003,288,4186,"MULTIPOLYGON (((8938088.457 2157739.160, 89380...",0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,2021S051610010004,438,9064,"POLYGON ((8976008.480 2163749.357, 8976015.274...",0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,2021S051610010005,203,1441,"POLYGON ((8969716.249 2163377.051, 8969785.883...",2,5,1,3,1,3,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,11,356,"MULTIPOLYGON (((5263454.031 3662224.177, 52634...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5429,2021S051662080005,13,414,"MULTIPOLYGON (((6043125.089 3568426.163, 60431...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5430,2021S051662080006,8,295,"MULTIPOLYGON (((6280121.289 3558103.571, 62801...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5431,2021S051662080007,0,0,"MULTIPOLYGON (((5544792.883 3549525.897, 55447...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# STEP 7: Processing it into Centroids for Counts and Choropleth for Rates

Since the earlier table shows the *weighted* numbers of directly exposed businesses and employees (by work location) together with *total* number of employees (by work location) for each affected NAICS code, the former is separated from the latter. The former needs to be processed into separate GDFs for centroids (to show counts) and choropleths (to show percentages)

In [23]:
# ============================================================
# SEPARATE WEIGHTED COLUMNS FROM NAICS COLUMNS - DYNAMIC
# ============================================================

# Build excluded columns list dynamically
excluded_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    excluded_cols.extend([f'{prefix}_B', f'{prefix}_E'])
excluded_cols.append('geometry')

business_filter = business_grouped[['ADADGUID'] + [col for col in business_grouped.columns if col in excluded_cols]]

business_census = business_grouped[[col for col in business_grouped.columns if col not in excluded_cols]]

In [24]:
# Convert to GeoDataFrame
cent_gdf = gpd.GeoDataFrame(business_filter, geometry='geometry', crs = 'EPSG:3347')

# Set a point within each polygon
cent_gdf = cent_gdf.drop(columns=['All_Businesses', 'All_Employees'])
cent_gdf['geometry'] = cent_gdf.geometry.representative_point()
cent_gdf.set_geometry('geometry', inplace=True)
cent_gdf

,ADADGUID,geometry,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,Cop_E,...,Section 338_B,Section 338_E,CUSMA_B,CUSMA_E,Total_B,Total_E,ScenBefore_B,ScenBefore_E,ScenAfter_B,ScenAfter_E
0,2021S051610010001,POINT (8927316.642 2156398.591),0,0,1,1,1,1,0,0,...,9,31,16,845,16,845,16,845,16,845
1,2021S051610010002,POINT (8966678.602 2164982.461),0,0,0,0,0,0,0,0,...,2,10,2,10,2,10,2,10,2,10
2,2021S051610010003,POINT (8934606.046 2144277.889),0,0,1,1,1,1,0,0,...,6,14,9,54,9,54,9,54,9,54
3,2021S051610010004,POINT (8976138.327 2157799.904),0,0,1,1,1,1,1,1,...,5,17,7,62,7,62,7,62,7,62
4,2021S051610010005,POINT (8970965.684 2157632.700),2,5,1,3,1,3,0,0,...,2,4,5,35,5,35,5,35,5,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,POINT (5259010.983 3653721.631),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5429,2021S051662080005,POINT (6041286.124 3573109.546),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5430,2021S051662080006,POINT (6281761.356 3557612.071),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5431,2021S051662080007,POINT (5550011.303 3546406.473),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
# ============================================================
# CHOROPLETH PERCENTAGES - DYNAMIC CATEGORIES
# ============================================================

choro_cols = business_filter.copy()

# Use all category prefixes dynamically
tars = all_category_prefixes

for tar in tars:
    tar_1 = f'{tar}_1'
    tar_2 = f'{tar}_2'

    choro_cols[tar_1] = (
        choro_cols[f'{tar}_B']/choro_cols['All_Businesses']
    )

    choro_cols[tar_2] = (
        choro_cols[f'{tar}_E']/choro_cols['All_Employees']
    )

choro_cols = choro_cols[['ADADGUID'] + [f'{tar}_1' for tar in tars] + [f'{tar}_2' for tar in tars] + ['geometry']]

choro_gdf = gpd.GeoDataFrame(choro_cols, geometry = 'geometry', crs = 'EPSG:3347')
choro_gdf

,ADADGUID,Auto_1,Alum_1,Steel_1,Cop_1,Ene_1,MHDV_1,LumOld_1,LumNew_1,Dairy_1,...,Alcohol_2,Motor_2,before August 22_2,after August 22_2,Section 338_2,CUSMA_2,Total_2,ScenBefore_2,ScenAfter_2,geometry
0,2021S051610010001,0.000000,0.004566,0.004566,0.000000,0.000000,0.0,0.009132,0.013699,0.0,...,0.000000,0.010653,0.004811,0.013058,0.010653,0.290378,0.290378,0.290378,0.290378,"MULTIPOLYGON (((8921559.157 2130255.903, 89215..."
1,2021S051610010002,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.025381,0.000000,0.025381,0.025381,0.025381,0.025381,0.025381,0.025381,"POLYGON ((8959571.943 2171799.486, 8959576.689..."
2,2021S051610010003,0.000000,0.003472,0.003472,0.000000,0.000000,0.0,0.013889,0.013889,0.0,...,0.000239,0.003106,0.010272,0.012422,0.003344,0.012900,0.012900,0.012900,0.012900,"MULTIPOLYGON (((8938088.457 2157739.160, 89380..."
3,2021S051610010004,0.000000,0.002283,0.002283,0.002283,0.004566,0.0,0.000000,0.002283,0.0,...,0.000000,0.001876,0.005075,0.006840,0.001876,0.006840,0.006840,0.006840,0.006840,"POLYGON ((8976008.480 2163749.357, 8976015.274..."
4,2021S051610010005,0.009852,0.004926,0.004926,0.000000,0.000000,0.0,0.000000,0.009852,0.0,...,0.000000,0.002776,0.023595,0.024289,0.002776,0.024289,0.024289,0.024289,0.024289,"POLYGON ((8969716.249 2163377.051, 8969785.883..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((5263454.031 3662224.177, 52634..."
5429,2021S051662080005,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((6043125.089 3568426.163, 60431..."
5430,2021S051662080006,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((6280121.289 3558103.571, 62801..."
5431,2021S051662080007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((5544792.883 3549525.897, 55447..."


# STEP 8: Generating Weight of Directly Exposed Jobs to Total Accessible Jobs in Each Industry

It is likely that while employees live close to their workplace, they do not live in the same ADA as they work in.  
  
StatsCan Census 2021 data shows a huge drop in the number of Canadians who travel more than 15km to their work vis-a-vis those who travel less than that distance to work.  
  
Thus, this cell creates a dictionary where for each ADA, it lists down, including itself, the ADA IDs within a 15km buffer around it (for small ADAs) or ADA IDs that are adjacent to it (for large ADAs). Small ADAs are defined as ADAs with an area less than (15km)^2.

In [26]:
# # Copy from earlier ADA shapefile and ensure it's in projected CRS (EPSG:3347)
# adas = ada.to_crs("EPSG:3347")

# # Create a column to mark if ADA is "small" (<= 706 km2)
# adas['is_small'] = adas['LANDAREA'] <= 706

# # Build spatial index once
# adas_sindex = adas.sindex

# # Simplify geometries early to reduce memory use
# adas['geometry'] = adas['geometry'].simplify(100)

# # Prepare empty dictionary
# ada_neighbors = {}

# # Process in chunks to avoid RAM overload
# chunk_size = 500
# n = len(adas)

# # Track overall time
# start_time = time.time()

# for start in tqdm(range(0, n, chunk_size)):
#     chunk_start_time = time.time()
#     end = min(start + chunk_size, n)
#     chunk = adas.iloc[start:end].copy()

#     for idx, row in chunk.iterrows():
#         ada_uid = row['DGUID']
#         geom = row['geometry']
#         is_small = row['is_small']

#         if is_small:
#             buffer_geom = geom.buffer(15000)
#             possible_matches_index = list(adas_sindex.intersection(buffer_geom.bounds))
#             possible_matches = adas.iloc[possible_matches_index]
#             matches = possible_matches[possible_matches.geometry.intersects(buffer_geom)]
#         else:
#             possible_matches_index = list(adas_sindex.intersection(geom.bounds))
#             possible_matches = adas.iloc[possible_matches_index]
#             matches = possible_matches[possible_matches.geometry.touches(geom) | (possible_matches['DGUID'] == ada_uid)]

#         ada_neighbors[ada_uid] = matches['DGUID'].tolist()

#     del chunk
#     gc.collect()

#     # Print chunk timing
#     chunk_elapsed = time.time() - chunk_start_time
#     print(f"Chunk {start}-{end} processed in {timedelta(seconds=chunk_elapsed)}")

# # Overall timing
# total_elapsed = time.time() - start_time
# print(f"\nTotal time taken: {timedelta(seconds=total_elapsed)}")

# # Optionally save the dictionary to disk
# with open("ada_neighbors.json", "w") as f:
#     json.dump(ada_neighbors, f)

In [27]:
with open('ada_neighbors.json') as f:
    ada_neighbors = json.load(f)

The dictionary is then used in conjunction with the *total* count of employees (by work location), as separated in Cell 13 above, to find out the likely number of jobs of each 6-digit NAICS, and total number of jobs, that are 'accessible' from each ADA --> going by the assumption of travel distance made by Canadians to go to work from Census 2021 data

In [28]:
# Ensure 'ADADGUID' is the index for fast lookup
business_census_indexed = business_census.set_index('ADADGUID')

# Prepare list to collect results
aggregated_results = []

# Loop through ADA + its neighbors
for ada_id, neighbor_list in tqdm(ada_neighbors.items()):
    # Filter business_census rows for all neighbors
    rows = business_census_indexed.loc[business_census_indexed.index.intersection(neighbor_list)]
    
    # Sum across all rows (by column)
    summed = rows.sum()
    
    # Store result with ADA ID
    result = summed.to_dict()
    result['ADADGUID'] = ada_id
    
    aggregated_results.append(result)

# Convert to DataFrame
jobs = pd.DataFrame(aggregated_results)

100%|██████████| 5433/5433 [00:03<00:00, 1771.77it/s]


This is to find out the rate of directly exposed jobs (6-digit NAICS) to total jobs in each industry (1-digit NAICS jobs) that are accessible from each ADA

In [29]:
jobs_rate = jobs.copy()

cola = [col for col in jobs.columns if col != 'ADADGUID']

# Calculate summed groups by first digit of column name
jobs_rate['Sum1'] = jobs_rate[[col for col in cola if col.startswith('1')]].sum(axis=1)
jobs_rate['Sum2'] = jobs_rate[[col for col in cola if col.startswith('2')]].sum(axis=1)
jobs_rate['Sum3'] = jobs_rate[[col for col in cola if col.startswith('3')]].sum(axis=1)

# Compute share per column
for col in cola:
    col_rate = f'{col}_R'
    if col.startswith('1'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum1']
    elif col.startswith('2'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum2']
    elif col.startswith('3'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
    else:
        jobs_rate[col_rate] = 0  # fallback in case of unexpected prefix

# Final filtered DataFrame: only ADADGUID and the *_R columns
jobs_rate = jobs_rate[['ADADGUID'] + [f'{col}_R' for col in cola]]

jobs_rate = jobs_rate.fillna(0)

jobs_rate

C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\3702249403.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\3702249403.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\3702249403.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

,ADADGUID,316110_R,337214_R,335315_R,325320_R,311824_R,112920_R,212397_R,332321_R,112510_R,...,111130_R,316210_R,212114_R,333130_R,339950_R,333413_R,334210_R,311351_R,326191_R,323116_R
0,2021S051610010001,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0
1,2021S051610010002,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.013755,...,0.004127,0.0,0.0,0.026483,0.033192,0.0,0.0,0.002472,0.0,0.0
2,2021S051610010003,0.010201,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0
3,2021S051610010004,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012376,...,0.003713,0.0,0.0,0.023885,0.024947,0.0,0.0,0.001858,0.0,0.0
4,2021S051610010005,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012376,...,0.003713,0.0,0.0,0.023885,0.024947,0.0,0.0,0.001858,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0
5429,2021S051662080005,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0
5430,2021S051662080006,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0
5431,2021S051662080007,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0


# STEP 9: Applying Jobs Weight to Census Data

Census 2021 Data reports residents' occupational NAICS code at the two-digit level

In [30]:
census = pd.read_csv('../input-data/large_size_data/98-401-X2021012_English_CSV_data.csv', encoding='latin1')

census = census[census['CHARACTERISTIC_ID'].isin([2259, 2262, 2263, 2266])] # Only taking the relevant 2-digit NAICS codes from Census 2021 data

census['ADADGUID'] = census['DGUID']

census['ProvinceCode'] = census['ADADGUID'].str[9:11].astype(int)

census['Province'] = census['ProvinceCode'].map(province_code)

census['CHARACTERISTIC_NAME'] = (
    census['CHARACTERISTIC_NAME']
    .str.replace(' ', '', regex=False)
    .str[:2]
)

census = census[['ADADGUID', 'Province', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']]

census_pivot = census.pivot_table(
    index=['ADADGUID', 'Province'],
    columns='CHARACTERISTIC_NAME',
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

census_pivot

CHARACTERISTIC_NAME,ADADGUID,Province,11,21,31,To
0,2021S051610010001,NL,455.0,120.0,665.0,3715.0
1,2021S051610010002,NL,50.0,70.0,60.0,2120.0
2,2021S051610010003,NL,160.0,100.0,310.0,4530.0
3,2021S051610010004,NL,20.0,170.0,130.0,5145.0
4,2021S051610010005,NL,35.0,215.0,130.0,5190.0
...,...,...,...,...,...,...
4957,2021S051662080002,NU,10.0,15.0,10.0,860.0
4958,2021S051662080003,NU,0.0,0.0,0.0,210.0
4959,2021S051662080004,NU,10.0,0.0,0.0,510.0
4960,2021S051662080005,NU,10.0,10.0,0.0,465.0


Thus the weights from Step 8 is used to estimate how many employees (by primary residence) are working in industries directly exposed to tariffs

In [31]:
# Merge on ADADGUID to align both datasets
merged = census_pivot.merge(jobs_rate, on='ADADGUID', how='right')  # or 'left' if census is base

# Start building the adjusted DataFrame
adjusted_jobs = merged[['ADADGUID', 'Province', 'To']].copy()

# Compute adjusted values
for col in cola:
    rate_col = f'{col}_R'
    prefix = col[:2]

    if prefix in ['21', '22']:
        source_col = '21'
    elif prefix in ['31', '32', '33']:
        source_col = '31'
    else:
        source_col = prefix

    adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])

adjusted_jobs['Sum'] = adjusted_jobs.drop(columns=['ADADGUID', 'Province', 'To']).sum(axis=1)

adjusted_jobs = adjusted_jobs.fillna(0)

adjusted_jobs['Province'] = adjusted_jobs['Province'].mask(
    adjusted_jobs['Province'].isin([0, np.nan]),
    adjusted_jobs['ADADGUID'].str[9:11].astype(int).map(province_code)
)

adjusted_jobs

C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\2488107975.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\2488107975.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\2488107975.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

,ADADGUID,Province,To,316110,337214,335315,325320,311824,112920,212397,...,316210,212114,333130,339950,333413,334210,311351,326191,323116,Sum
0,2021S051610010001,NL,3715.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1129.0
1,2021S051610010002,NL,2120.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,2.0,0.0,0.0,1.0,0.0,0.0,225.0
2,2021S051610010003,NL,4530.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,590.0
3,2021S051610010004,NL,5145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,4.0,4.0,0.0,0.0,1.0,0.0,0.0,379.0
4,2021S051610010005,NL,5190.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,4.0,4.0,0.0,0.0,1.0,0.0,0.0,437.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,NU,510.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5429,2021S051662080005,NU,465.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5430,2021S051662080006,NU,280.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5431,2021S051662080007,NU,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 10: Applying Export Weights to the Census Data

Export weights from Step 4 are applied to account for regional differences

In [32]:
lw_sb = weight_naics_sb.melt(id_vars='NAICS', var_name='Province', value_name='Rate')
lw_sb['NAICS'] = lw_sb['NAICS'].astype(str) + '_r'
weight_naics_pivot_sb = lw_sb.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

In [33]:
# Step 1: Melt to long format
long_weight = weight_naics.melt(id_vars='NAICS', var_name='Province', value_name='Rate')

long_weight['NAICS'] = long_weight['NAICS'].astype(str) + '_r'

# Step 2: Pivot to wide format
weight_naics_pivot = long_weight.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

weight_naics_pivot

NAICS,Province,111110_r,111120_r,111130_r,111140_r,111150_r,111160_r,111190_r,111211_r,111219_r,...,337215_r,337910_r,337920_r,339110_r,339910_r,339920_r,339930_r,339940_r,339950_r,339990_r
0,AL,0.024138,0.034204,0.0,0.032376,0.370000,0.000000,0.117974,0.0,0.042798,...,0.987545,0.514049,0.777591,0.532096,0.567977,0.944934,0.877695,0.720095,0.206830,0.614544
1,BC,0.000000,0.028733,0.0,0.028748,0.243865,0.000000,0.156439,0.0,0.000049,...,0.887124,0.186107,0.872506,0.590677,0.823359,0.742929,0.606863,0.691042,0.265313,0.375425
2,MB,0.064938,0.094353,0.0,0.030494,0.370000,0.000000,0.264963,0.0,0.205623,...,0.996193,0.790000,0.489556,0.450144,0.336387,0.832748,0.802942,0.859608,0.151730,0.738672
3,NB,0.620000,0.124930,0.0,0.620000,0.000000,0.000000,0.000000,0.0,0.006528,...,0.998284,0.790000,0.205144,0.849244,0.981112,0.272781,1.000000,0.831198,0.699527,0.406811
4,NL,0.000000,0.620000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.956908,0.000000,0.000000,0.416990,0.290542,0.079019,0.854647,0.000000,0.000000,0.133558
5,NS,0.000000,0.000000,0.0,0.620000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.914146,0.000000,0.000000,0.703223,0.821844,0.949896,0.945270,0.049719,0.149934,0.629227
6,NU,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.870000,1.000000,0.000000,1.000000,0.000000,0.150000,1.000000
7,NWT,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.981535,0.000000,0.000000,0.000000,0.000000,0.000000
8,ON,0.057076,0.371635,0.0,0.032567,0.074227,0.315716,0.206290,0.0,0.001741,...,0.964217,0.728146,0.436047,0.555835,0.912029,0.702178,0.591745,0.732703,0.562703,0.815254
9,PEI,0.000000,0.620000,0.0,0.000000,0.000000,0.000000,0.370000,0.0,0.000000,...,0.106944,0.000000,0.000000,0.000000,0.989954,0.227019,0.000000,0.000000,0.150000,0.830635


In [34]:
# census by jobs for sb

census_byjobs_sb = adjusted_jobs.merge(weight_naics_pivot_sb, on='Province', how='left')
for col in cola:
    census_byjobs_sb[col] = np.ceil(census_byjobs_sb[col] * census_byjobs_sb[f'{col}_r'])

In [35]:
# ============================================================
# APPLY EXPORT WEIGHTS - DYNAMIC CATEGORIES
# ============================================================

# Step 1: Merge adjusted_jobs with weight_naics_pivot on Province
census_byjobs = adjusted_jobs.merge(weight_naics_pivot, on='Province', how='left')

# Step 2: Multiply each column by its corresponding rate
for col in cola:
    rate_col = f'{col}_r'
    
    census_byjobs[col] = np.ceil(census_byjobs[col] * census_byjobs[rate_col])

# Step 3: Keep only ADADGUID and updated values
census_byjobs = census_byjobs[['ADADGUID', 'To'] + cola]

# Define output dictionary dynamically
grouped_data = {
    'ADADGUID': census_byjobs['ADADGUID'],  # retain ADA ID
    'Census': census_byjobs['To'],
}

# Add category sums dynamically
for prefix, naics_set in category_naics.items():
    src = census_byjobs_sb if prefix == 'ScenBefore' else census_byjobs
    grouped_data[f'{prefix}_C'] = src[[col for col in cola if col in naics_set]].sum(axis=1)

# Create final grouped DataFrame
census_bytariffs = pd.DataFrame(grouped_data)
census_bytariffs

,ADADGUID,Census,Auto_C,Alum_C,Steel_C,Cop_C,Ene_C,MHDV_C,LumOld_C,LumNew_C,Dairy_C,Alcohol_C,Motor_C,before August 22_C,after August 22_C,Section 338_C,CUSMA_C,Total_C,ScenBefore_C,ScenAfter_C
0,2021S051610010001,3715.0,0.0,1.0,1.0,0.0,0.0,0.0,5.0,8.0,0.0,0.0,165.0,9.0,168.0,165.0,613.0,613.0,613.0,613.0
1,2021S051610010002,2120.0,6.0,8.0,8.0,1.0,7.0,1.0,3.0,8.0,2.0,1.0,48.0,25.0,62.0,48.0,82.0,82.0,82.0,82.0
2,2021S051610010003,4530.0,0.0,1.0,2.0,0.0,28.0,0.0,8.0,10.0,1.0,3.0,81.0,39.0,116.0,82.0,258.0,258.0,258.0,258.0
3,2021S051610010004,5145.0,6.0,9.0,8.0,1.0,13.0,1.0,4.0,18.0,2.0,1.0,35.0,42.0,63.0,35.0,98.0,98.0,98.0,98.0
4,2021S051610010005,5190.0,6.0,9.0,8.0,1.0,15.0,1.0,4.0,18.0,2.0,1.0,42.0,44.0,72.0,42.0,110.0,110.0,110.0,110.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,510.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5429,2021S051662080005,465.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5430,2021S051662080006,280.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5431,2021S051662080007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 11: Processing and adding the data to Choropleth and Centroid GDFs

Add the data on employees (by primary residence) to the GDFs produced in Step 7

In [36]:
# ============================================================
# CENTROIDS EXPORT - DYNAMIC CATEGORIES
# ============================================================

centroids = cent_gdf.merge(census_bytariffs, on='ADADGUID', how='left')

# Build column list dynamically
centroid_cols = ['ADADGUID']
for tar in tars:
    centroid_cols.extend([f'{tar}_B', f'{tar}_E', f'{tar}_C'])
centroid_cols.append('geometry')

centroids = centroids[centroid_cols]
centroids = centroids.to_crs('EPSG:4326')
centroids.to_file('centroids.geojson', driver='GeoJSON')
centroids.to_csv("centroids.csv", index=False)
centroids

,ADADGUID,Auto_B,Auto_E,Auto_C,Alum_B,Alum_E,Alum_C,Steel_B,Steel_E,Steel_C,...,Total_B,Total_E,Total_C,ScenBefore_B,ScenBefore_E,ScenBefore_C,ScenAfter_B,ScenAfter_E,ScenAfter_C,geometry
0,2021S051610010001,0,0,0.0,1,1,1.0,1,1,1.0,...,16,845,613.0,16,845,613.0,16,845,613.0,POINT (-53.25419 47.86225)
1,2021S051610010002,0,0,6.0,0,0,8.0,0,0,8.0,...,2,10,82.0,2,10,82.0,2,10,82.0,POINT (-52.76054 47.72309)
2,2021S051610010003,0,0,0.0,1,1,1.0,1,1,2.0,...,9,54,258.0,9,54,258.0,9,54,258.0,POINT (-53.26649 47.73593)
3,2021S051610010004,0,0,6.0,1,1,9.0,1,1,8.0,...,7,62,98.0,7,62,98.0,7,62,98.0,POINT (-52.71312 47.62177)
4,2021S051610010005,2,5,6.0,1,3,9.0,1,3,8.0,...,5,35,110.0,5,35,110.0,5,35,110.0,POINT (-52.77030 47.64725)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-115.37130 67.80897)
5429,2021S051662080005,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-95.88322 68.64144)
5430,2021S051662080006,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-89.80822 68.53258)
5431,2021S051662080007,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-107.82111 67.68532)


In [37]:
# ============================================================
# PERCENTAGE BY TARIFFS - DYNAMIC CATEGORIES
# ============================================================

perc_bytariffs = census_bytariffs.copy()

for tar in tars:
    tar_3 = f'{tar}_3'

    perc_bytariffs[tar_3] = (
        perc_bytariffs[f'{tar}_C']/perc_bytariffs['Census']
    )

perc_bytariffs = perc_bytariffs[['ADADGUID'] + [f'{tar}_3' for tar in tars]]
# # Clip values to max 1 (100%) for CUSMA and Total
# perc_bytariffs['CUSMA_3'] = perc_bytariffs['CUSMA_3'].clip(upper=1)
# perc_bytariffs['Total_3'] = perc_bytariffs['Total_3'].clip(upper=1)

for p in ['CUSMA', 'Total', 'ScenBefore', 'ScenAfter']:
    perc_bytariffs[f'{p}_3'] = perc_bytariffs[f'{p}_3'].clip(upper=1)
perc_bytariffs

,ADADGUID,Auto_3,Alum_3,Steel_3,Cop_3,Ene_3,MHDV_3,LumOld_3,LumNew_3,Dairy_3,Alcohol_3,Motor_3,before August 22_3,after August 22_3,Section 338_3,CUSMA_3,Total_3,ScenBefore_3,ScenAfter_3
0,2021S051610010001,0.000000,0.000269,0.000269,0.000000,0.000000,0.000000,0.001346,0.002153,0.000000,0.000000,0.044415,0.002423,0.045222,0.044415,0.165007,0.165007,0.165007,0.165007
1,2021S051610010002,0.002830,0.003774,0.003774,0.000472,0.003302,0.000472,0.001415,0.003774,0.000943,0.000472,0.022642,0.011792,0.029245,0.022642,0.038679,0.038679,0.038679,0.038679
2,2021S051610010003,0.000000,0.000221,0.000442,0.000000,0.006181,0.000000,0.001766,0.002208,0.000221,0.000662,0.017881,0.008609,0.025607,0.018102,0.056954,0.056954,0.056954,0.056954
3,2021S051610010004,0.001166,0.001749,0.001555,0.000194,0.002527,0.000194,0.000777,0.003499,0.000389,0.000194,0.006803,0.008163,0.012245,0.006803,0.019048,0.019048,0.019048,0.019048
4,2021S051610010005,0.001156,0.001734,0.001541,0.000193,0.002890,0.000193,0.000771,0.003468,0.000385,0.000193,0.008092,0.008478,0.013873,0.008092,0.021195,0.021195,0.021195,0.021195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5429,2021S051662080005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5430,2021S051662080006,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5431,2021S051662080007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# ============================================================
# CHOROPLETH EXPORT - DYNAMIC CATEGORIES
# ============================================================

choropleth = choro_gdf.merge(perc_bytariffs, on='ADADGUID', how='right')

# Build column list dynamically
choro_export_cols = ['ADADGUID']
for tar in tars:
    choro_export_cols.extend([f'{tar}_1', f'{tar}_2', f'{tar}_3'])
choro_export_cols.append('geometry')

choropleth = choropleth[choro_export_cols]
choropleth = choropleth.to_crs('EPSG:4326')
choropleth

,ADADGUID,Auto_1,Auto_2,Auto_3,Alum_1,Alum_2,Alum_3,Steel_1,Steel_2,Steel_3,...,Total_1,Total_2,Total_3,ScenBefore_1,ScenBefore_2,ScenBefore_3,ScenAfter_1,ScenAfter_2,ScenAfter_3,geometry
0,2021S051610010001,0.000000,0.00000,0.000000,0.004566,0.000344,0.000269,0.004566,0.000344,0.000269,...,0.073059,0.290378,0.165007,0.073059,0.290378,0.165007,0.073059,0.290378,0.165007,"MULTIPOLYGON (((-53.51451 47.69912, -53.51464 ..."
1,2021S051610010002,0.000000,0.00000,0.002830,0.000000,0.000000,0.003774,0.000000,0.000000,0.003774,...,0.030303,0.025381,0.038679,0.030303,0.025381,0.038679,0.030303,0.025381,0.038679,"POLYGON ((-52.78543 47.80961, -52.78542 47.809..."
2,2021S051610010003,0.000000,0.00000,0.000000,0.003472,0.000239,0.000221,0.003472,0.000239,0.000442,...,0.031250,0.012900,0.056954,0.031250,0.012900,0.056954,0.031250,0.012900,0.056954,"MULTIPOLYGON (((-53.12645 47.81702, -53.12663 ..."
3,2021S051610010004,0.000000,0.00000,0.001166,0.002283,0.000110,0.001749,0.002283,0.000110,0.001555,...,0.015982,0.006840,0.019048,0.015982,0.006840,0.019048,0.015982,0.006840,0.019048,"POLYGON ((-52.66902 47.66588, -52.66898 47.665..."
4,2021S051610010005,0.009852,0.00347,0.001156,0.004926,0.002082,0.001734,0.004926,0.002082,0.001541,...,0.024631,0.024289,0.021195,0.024631,0.024289,0.021195,0.024631,0.024289,0.021195,"POLYGON ((-52.73992 47.69568, -52.74026 47.694..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5428,2021S051662080004,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((-115.34503 67.89695, -115.3453..."
5429,2021S051662080005,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((-95.82942 68.59941, -95.82932 ..."
5430,2021S051662080006,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((-89.84909 68.53759, -89.84931 ..."
5431,2021S051662080007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-107.96280 67.70120, -107.9628..."


In [39]:
choropleth.to_file('choropleth.geojson', driver='GeoJSON')
choropleth.to_file('choropleth.shp', driver='ESRI Shapefile')

C:\Users\yihoi\AppData\Local\Temp\ipykernel_5004\4259686084.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  choropleth.to_file('choropleth.shp', driver='ESRI Shapefile')


In [40]:
choropleth.drop(columns="geometry").to_csv("choropleth.csv", index=False)

# CSV Trails

for double-checking purposes

In [41]:
business_grouped.drop(columns='geometry').to_csv('trail.csv', index=False)
business_filter.drop(columns='geometry').to_csv('trail2.csv', index=False)
business_census.to_csv('trail3.csv', index=False)

In [42]:
choro_cols.drop(columns='geometry').to_csv('trail4.csv', index=False)

In [43]:
jobs.to_csv('trail5.csv')
jobs_rate.to_csv('trail6.csv')
adjusted_jobs.to_csv('trail7.csv')